xgboost

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 18.2 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Train-test split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for XGBoost
def objective(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 0, 5),
        reg_lambda=trial.suggest_float("reg_lambda", 0, 5),
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
best_model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='mlogloss', random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# Compute AUC
try:
    if y_prob.shape[1] == 2:
        auc = roc_auc_score(y_test, y_prob[:, 1])
    else:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save results
df_result = pd.DataFrame([metrics])
df_result.to_csv("xgboost_performance_for_dpc.csv", index=False)
print(df_result)

# Download file (for Colab)
from google.colab import files
files.download('xgboost_performance_for_dpc.csv')

[I 2025-06-13 04:38:08,368] A new study created in memory with name: no-name-7187fea0-57d0-4967-b7ee-ad66aae2d2c1
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:38:11] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-06-13 04:38:42,268] Trial 0 finished with value: 0.8639747995418099 and parameters: {'n_estimators': 89, 'max_depth': 13, 'learning_rate': 0.1615326515592441, 'subsample': 0.7095445072696662, 'colsample_bytree': 0.6711613803449139, 'gamma': 2.9275859118045715, 'reg_alpha': 3.8867357245592027, 'reg_lambda': 2.6600649837791734}. Best is trial 0 with value: 0.8639747995418099.
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:38:43] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-06-13 04:39:27,128] Trial 1 finished with value: 0.8665521191294387 a

     Model  Accuracy  Precision    Recall  F1 Score  Specificity       AUC  \
0  XGBoost   0.87543   0.855769  0.808997  0.828052      0.94841  0.921426   

      Kappa       MCC  
0  0.657212  0.663119  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

extra_trees

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.ensemble import ExtraTreesClassifier

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Extra Trees
def objective(trial):
    model = ExtraTreesClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 30),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy"]),
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
best_model = ExtraTreesClassifier(**best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# Compute AUC
try:
    if y_prob.shape[1] == 2:
        auc = roc_auc_score(y_test, y_prob[:, 1])
    else:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "Extra Trees",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("extra_trees_performance_for_dpc.csv", index=False)
print(df_result)

# Download CSV (for Colab)
from google.colab import files
files.download('extra_trees_performance_for_dpc.csv')

[I 2025-06-13 05:26:54,098] A new study created in memory with name: no-name-20a8ab53-ce63-4fe3-93d4-5fe5a7ea6ef4
[I 2025-06-13 05:27:04,443] Trial 0 finished with value: 0.8021191294387171 and parameters: {'n_estimators': 264, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 16, 'criterion': 'entropy'}. Best is trial 0 with value: 0.8021191294387171.
[I 2025-06-13 05:27:05,830] Trial 1 finished with value: 0.7749140893470791 and parameters: {'n_estimators': 60, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 5, 'criterion': 'gini'}. Best is trial 0 with value: 0.8021191294387171.
[I 2025-06-13 05:27:10,865] Trial 2 finished with value: 0.8121420389461627 and parameters: {'n_estimators': 137, 'max_depth': 15, 'min_samples_split': 11, 'min_samples_leaf': 4, 'criterion': 'entropy'}. Best is trial 2 with value: 0.8121420389461627.
[I 2025-06-13 05:27:12,386] Trial 3 finished with value: 0.7926689576174112 and parameters: {'n_estimators': 53, 'max_depth': 12, 'min

         Model  Accuracy  Precision    Recall  F1 Score  Specificity  \
0  Extra Trees   0.83362   0.868198  0.694525  0.728161     0.986424   

        AUC     Kappa       MCC  
0  0.902264  0.474701  0.535251  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

gradient_boosting

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.ensemble import GradientBoostingClassifier

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Gradient Boosting
def objective(trial):
    model = GradientBoostingClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
best_model = GradientBoostingClassifier(**best_params, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# Compute AUC
try:
    if y_prob.shape[1] == 2:
        auc = roc_auc_score(y_test, y_prob[:, 1])
    else:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "Gradient Boosting",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("gradient_boosting_performance_for_dpc.csv", index=False)
print(df_result)

# Download CSV (for Colab)
from google.colab import files
files.download('gradient_boosting_performance_for_dpc.csv')


[I 2025-06-13 05:40:43,677] A new study created in memory with name: no-name-fd73c6dc-14be-4ced-af8c-8cadef692bbf
[I 2025-06-13 05:49:09,871] Trial 0 finished with value: 0.8651202749140894 and parameters: {'n_estimators': 129, 'learning_rate': 0.17940103805561577, 'max_depth': 13, 'subsample': 0.8209449479449753, 'min_samples_split': 20, 'min_samples_leaf': 18}. Best is trial 0 with value: 0.8651202749140894.
[I 2025-06-13 05:56:06,145] Trial 1 finished with value: 0.8628293241695304 and parameters: {'n_estimators': 176, 'learning_rate': 0.19049663457263588, 'max_depth': 9, 'subsample': 0.6391829007260788, 'min_samples_split': 16, 'min_samples_leaf': 13}. Best is trial 0 with value: 0.8651202749140894.
[I 2025-06-13 06:00:59,611] Trial 2 finished with value: 0.8697021764032073 and parameters: {'n_estimators': 275, 'learning_rate': 0.05955756932525868, 'max_depth': 4, 'subsample': 0.6892558844121013, 'min_samples_split': 13, 'min_samples_leaf': 19}. Best is trial 2 with value: 0.869702

               Model  Accuracy  Precision    Recall  F1 Score  Specificity  \
0  Gradient Boosting  0.871707    0.86702  0.787055  0.815281     0.964701   

        AUC     Kappa       MCC  
0  0.918608  0.633623  0.649169  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

knn

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.neighbors import KNeighborsClassifier

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for KNN
def objective(trial):
    model = KNeighborsClassifier(
        n_neighbors=trial.suggest_int("n_neighbors", 1, 30),
        weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
        metric=trial.suggest_categorical("metric", ["euclidean", "manhattan", "minkowski"])
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
best_model = KNeighborsClassifier(**best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# Compute AUC
try:
    if y_prob.shape[1] == 2:
        auc = roc_auc_score(y_test, y_prob[:, 1])
    else:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "KNN",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("knn_performance_for_dpc.csv", index=False)
print(df_result)

# Download CSV (for Colab)
from google.colab import files
files.download('knn_performance_for_dpc.csv')

[I 2025-06-13 13:21:05,989] A new study created in memory with name: no-name-18576c36-50fb-4673-8b7d-a1e7f2c39125
[I 2025-06-13 13:21:07,932] Trial 0 finished with value: 0.7517182130584192 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.7517182130584192.
[I 2025-06-13 13:21:38,185] Trial 1 finished with value: 0.8193012600229095 and parameters: {'n_neighbors': 11, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 1 with value: 0.8193012600229095.
[I 2025-06-13 13:21:39,784] Trial 2 finished with value: 0.7528636884306987 and parameters: {'n_neighbors': 6, 'weights': 'uniform', 'metric': 'minkowski'}. Best is trial 1 with value: 0.8193012600229095.
[I 2025-06-13 13:21:41,242] Trial 3 finished with value: 0.7101947308132875 and parameters: {'n_neighbors': 4, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 1 with value: 0.8193012600229095.
[I 2025-06-13 13:21:42,981] Trial 4 finished with value: 0.7657

  Model  Accuracy  Precision    Recall  F1 Score  Specificity       AUC  \
0   KNN  0.835338   0.787154  0.785728  0.786435     0.889837  0.882607   

      Kappa      MCC  
0  0.572872  0.57288  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

bagging

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 16.4 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier  # Default base estimator

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Bagging
def objective(trial):
    model = BaggingClassifier(
        # Changed 'base_estimator' to 'estimator'
        estimator=DecisionTreeClassifier(),
        n_estimators=trial.suggest_int("n_estimators", 10, 200),
        max_samples=trial.suggest_float("max_samples", 0.5, 1.0),
        max_features=trial.suggest_float("max_features", 0.5, 1.0),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

# Train best model
best_params = study.best_params
best_model = BaggingClassifier(
    # Changed 'base_estimator' to 'estimator' here as well
    estimator=DecisionTreeClassifier(),
    **best_params,
    random_state=42
)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# Compute AUC
try:
    if y_prob.shape[1] == 2:
        auc = roc_auc_score(y_test, y_prob[:, 1])
    else:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "Bagging",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("bagging_performance_for_dpc.csv", index=False)
print(df_result)

# Download CSV (for Colab)
from google.colab import files
files.download('bagging_performance_for_dpc.csv')

[I 2025-06-13 17:36:14,284] A new study created in memory with name: no-name-cd767724-8eac-447a-9487-ef39e31dd322
[I 2025-06-13 17:45:00,108] Trial 0 finished with value: 0.8485108820160366 and parameters: {'n_estimators': 193, 'max_samples': 0.6948353742917499, 'max_features': 0.6121001506537327, 'bootstrap': False}. Best is trial 0 with value: 0.8485108820160366.
[I 2025-06-13 17:48:21,782] Trial 1 finished with value: 0.8399198167239404 and parameters: {'n_estimators': 40, 'max_samples': 0.9362175572642204, 'max_features': 0.7969435432505998, 'bootstrap': False}. Best is trial 0 with value: 0.8485108820160366.
[I 2025-06-13 17:58:39,752] Trial 2 finished with value: 0.8465063001145475 and parameters: {'n_estimators': 135, 'max_samples': 0.7170853394509384, 'max_features': 0.9641605919945007, 'bootstrap': False}. Best is trial 0 with value: 0.8485108820160366.
[I 2025-06-13 18:02:15,890] Trial 3 finished with value: 0.8505154639175257 and parameters: {'n_estimators': 66, 'max_samples

     Model  Accuracy  Precision    Recall  F1 Score  Specificity       AUC  \
0  Bagging  0.855097   0.872386  0.741909  0.777293     0.979441  0.906613   

     Kappa       MCC  
0  0.56339  0.600278  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ridge

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 16.9 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.linear_model import RidgeClassifier

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Ridge Classifier
def objective(trial):
    model = RidgeClassifier(
        alpha=trial.suggest_float("alpha", 0.01, 10.0, log=True),
        solver=trial.suggest_categorical("solver", ["auto", "svd", "cholesky", "lsqr", "sparse_cg", "sag", "saga"]),
        tol=trial.suggest_float("tol", 1e-5, 1e-1, log=True)
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
best_model = RidgeClassifier(**best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# RidgeClassifier doesn't provide probabilities -> set AUC to 0
auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "Ridge Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("ridge_classifier_performance_for_dpc.csv", index=False)
print(df_result)

# Download CSV (for Colab)
from google.colab import files
files.download('ridge_classifier_performance_for_dpc.csv')

[I 2025-06-14 02:40:50,172] A new study created in memory with name: no-name-19b0842b-5d7f-44a2-9423-2c2a292f36fe
[I 2025-06-14 02:40:56,342] Trial 0 finished with value: 0.8513745704467354 and parameters: {'alpha': 0.057147256849259385, 'solver': 'saga', 'tol': 0.005084561755363755}. Best is trial 0 with value: 0.8513745704467354.
[I 2025-06-14 02:40:56,562] Trial 1 finished with value: 0.8508018327605956 and parameters: {'alpha': 0.03947500888492957, 'solver': 'auto', 'tol': 0.0008446107897961129}. Best is trial 0 with value: 0.8513745704467354.
[I 2025-06-14 02:40:56,731] Trial 2 finished with value: 0.8424971363115693 and parameters: {'alpha': 0.10249589446153957, 'solver': 'sparse_cg', 'tol': 0.08070010618124171}. Best is trial 0 with value: 0.8513745704467354.
[I 2025-06-14 02:40:56,951] Trial 3 finished with value: 0.8508018327605956 and parameters: {'alpha': 0.07041311552645153, 'solver': 'auto', 'tol': 0.000263059724259635}. Best is trial 0 with value: 0.8513745704467354.
[I 2

              Model  Accuracy  Precision    Recall  F1 Score  Specificity  \
0  Ridge Classifier  0.851947    0.83183  0.764139  0.788273      0.94841   

   AUC     Kappa       MCC  
0    0  0.579651  0.592112  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

voting

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Voting Classifier
def objective(trial):
    # Tune individual estimators
    svc = SVC(
        C=trial.suggest_float('svc_C', 0.1, 10.0),
        kernel=trial.suggest_categorical('svc_kernel', ['rbf', 'poly']),
        gamma='scale',
        probability=True,
        random_state=42
    )

    rf = RandomForestClassifier(
        n_estimators=trial.suggest_int('rf_n_estimators', 50, 200),
        max_depth=trial.suggest_int('rf_max_depth', 2, 20),
        random_state=42
    )

    lr = LogisticRegression(
        C=trial.suggest_float('lr_C', 0.01, 10.0),
        solver='lbfgs',
        max_iter=500,
        random_state=42
    )

    model = VotingClassifier(
        estimators=[
            ('svc', svc),
            ('rf', rf),
            ('lr', lr)
        ],
        voting='soft'
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Get best parameters
best_params = study.best_params

# Rebuild estimators with best parameters
svc = SVC(C=best_params['svc_C'], kernel=best_params['svc_kernel'], gamma='scale', probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=best_params['rf_n_estimators'], max_depth=best_params['rf_max_depth'], random_state=42)
lr = LogisticRegression(C=best_params['lr_C'], solver='lbfgs', max_iter=500, random_state=42)

best_model = VotingClassifier(
    estimators=[
        ('svc', svc),
        ('rf', rf),
        ('lr', lr)
    ],
    voting='soft'
)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# AUC calculation
try:
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "Voting Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("voting_classifier_performance_for_dpc.csv", index=False)
print(df_result)

# Download (Colab)
from google.colab import files
files.download('voting_classifier_performance_for_dpc.csv')

[I 2025-06-14 02:50:05,299] A new study created in memory with name: no-name-8f722340-1cf3-41ed-9900-5c09cf26649c
[I 2025-06-14 02:59:36,128] Trial 0 finished with value: 0.8490836197021764 and parameters: {'svc_C': 3.1176019034596227, 'svc_kernel': 'poly', 'rf_n_estimators': 86, 'rf_max_depth': 14, 'lr_C': 7.908290444414108}. Best is trial 0 with value: 0.8490836197021764.
[I 2025-06-14 03:11:24,584] Trial 1 finished with value: 0.847651775486827 and parameters: {'svc_C': 9.847086215414327, 'svc_kernel': 'poly', 'rf_n_estimators': 64, 'rf_max_depth': 12, 'lr_C': 4.914140397277196}. Best is trial 0 with value: 0.8490836197021764.
[I 2025-06-14 03:20:52,133] Trial 2 finished with value: 0.8754295532646048 and parameters: {'svc_C': 7.900135335187221, 'svc_kernel': 'rbf', 'rf_n_estimators': 178, 'rf_max_depth': 15, 'lr_C': 1.4839430723880658}. Best is trial 2 with value: 0.8754295532646048.
[I 2025-06-14 03:30:13,829] Trial 3 finished with value: 0.8748568155784651 and parameters: {'svc_C

               Model  Accuracy  Precision    Recall  F1 Score  Specificity  \
0  Voting Classifier  0.878007   0.868817  0.802621  0.827616     0.960822   

   AUC     Kappa       MCC  
0    0  0.657225  0.668167  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

logistic_regression

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Logistic Regression
def objective(trial):
    model = LogisticRegression(
        C=trial.suggest_float('C', 0.01, 10.0),
        solver=trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
        max_iter=trial.suggest_int('max_iter', 100, 500),
        penalty='l2',
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
best_model = LogisticRegression(
    **best_params,
    penalty='l2',
    random_state=42
)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# AUC
try:
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation
metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("logistic_regression_performance_for_dpc.csv", index=False)
print(df_result)

# Download (Colab)
from google.colab import files
files.download('logistic_regression_performance_for_dpc.csv')

[I 2025-06-14 11:58:00,318] A new study created in memory with name: no-name-f405b7f9-7005-4b0c-b38a-8ceed33df572
[I 2025-06-14 11:58:08,856] Trial 0 finished with value: 0.8490836197021764 and parameters: {'C': 9.963935657501073, 'solver': 'liblinear', 'max_iter': 216}. Best is trial 0 with value: 0.8490836197021764.
[I 2025-06-14 11:58:10,465] Trial 1 finished with value: 0.8482245131729668 and parameters: {'C': 9.299120379364604, 'solver': 'lbfgs', 'max_iter': 412}. Best is trial 0 with value: 0.8490836197021764.
[I 2025-06-14 11:58:12,295] Trial 2 finished with value: 0.8479381443298969 and parameters: {'C': 6.354372503064842, 'solver': 'lbfgs', 'max_iter': 117}. Best is trial 0 with value: 0.8490836197021764.
[I 2025-06-14 11:58:20,589] Trial 3 finished with value: 0.8493699885452463 and parameters: {'C': 8.460619702159942, 'solver': 'liblinear', 'max_iter': 391}. Best is trial 3 with value: 0.8493699885452463.
[I 2025-06-14 11:58:24,764] Trial 4 finished with value: 0.84936998854

                 Model  Accuracy  Precision    Recall  F1 Score  Specificity  \
0  Logistic Regression  0.849943   0.814164  0.782555  0.795973     0.923972   

   AUC     Kappa       MCC  
0    0  0.592737  0.595881  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

stacking_classifier

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 17.9 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Load dataset
df = pd.read_csv("/content/dpc_features_dataset.csv")
X = df.drop(columns=["label"])
y = df["label"]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Optuna objective for Stacking Classifier
def objective(trial):
    svc = SVC(
        C=trial.suggest_float("svc_C", 0.1, 10.0),
        kernel=trial.suggest_categorical("svc_kernel", ["rbf", "poly"]),
        gamma="scale",
        probability=True,
        random_state=42
    )

    rf = RandomForestClassifier(
        n_estimators=trial.suggest_int("rf_n_estimators", 50, 200),
        max_depth=trial.suggest_int("rf_max_depth", 2, 20),
        random_state=42
    )

    lr_base = LogisticRegression(
        C=trial.suggest_float("lr_base_C", 0.01, 10.0),
        solver="lbfgs",
        max_iter=500,
        random_state=42
    )

    lr_final = LogisticRegression(
        C=trial.suggest_float("lr_final_C", 0.01, 10.0),
        solver="lbfgs",
        max_iter=500,
        random_state=42
    )

    model = StackingClassifier(
        estimators=[
            ('svc', svc),
            ('rf', rf),
            ('lr_base', lr_base)
        ],
        final_estimator=lr_final,
        passthrough=True
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

# Rebuild with best params
best_params = study.best_params

svc = SVC(C=best_params["svc_C"], kernel=best_params["svc_kernel"], gamma="scale", probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=best_params["rf_n_estimators"], max_depth=best_params["rf_max_depth"], random_state=42)
lr_base = LogisticRegression(C=best_params["lr_base_C"], solver="lbfgs", max_iter=500, random_state=42)
lr_final = LogisticRegression(C=best_params["lr_final_C"], solver="lbfgs", max_iter=500, random_state=42)

best_model = StackingClassifier(
    estimators=[
        ('svc', svc),
        ('rf', rf),
        ('lr_base', lr_base)
    ],
    final_estimator=lr_final,
    passthrough=True
)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

# AUC calculation
try:
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
except:
    auc = 0

# Specificity
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

# Evaluation metrics
metrics = {
    "Model": "Stacking Classifier",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='macro'),
    "F1 Score": f1_score(y_test, y_pred, average='macro'),
    "Specificity": specificity,
    "AUC": auc,
    "Kappa": cohen_kappa_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

# Save result
df_result = pd.DataFrame([metrics])
df_result.to_csv("stacking_classifier_performance_for_dpc.csv", index=False)
print(df_result)

# Download (Colab)
from google.colab import files
files.download('stacking_classifier_performance_for_dpc.csv')

[I 2025-06-18 05:12:34,343] A new study created in memory with name: no-name-ca873acf-e9a7-4c5d-a300-c19153f5ef46
[I 2025-06-18 05:26:25,532] Trial 0 finished with value: 0.8820160366552119 and parameters: {'svc_C': 1.0980165835434461, 'svc_kernel': 'rbf', 'rf_n_estimators': 150, 'rf_max_depth': 16, 'lr_base_C': 1.7059135833269752, 'lr_final_C': 9.624206237494208}. Best is trial 0 with value: 0.8820160366552119.
[I 2025-06-18 05:45:51,759] Trial 1 finished with value: 0.8602520045819015 and parameters: {'svc_C': 1.294820121316951, 'svc_kernel': 'poly', 'rf_n_estimators': 198, 'rf_max_depth': 9, 'lr_base_C': 2.344954096704468, 'lr_final_C': 8.620454296243679}. Best is trial 0 with value: 0.8820160366552119.
[I 2025-06-18 06:06:35,350] Trial 2 finished with value: 0.8825887743413516 and parameters: {'svc_C': 5.0667569636078795, 'svc_kernel': 'rbf', 'rf_n_estimators': 74, 'rf_max_depth': 4, 'lr_base_C': 9.194185060324438, 'lr_final_C': 1.8828673874600044}. Best is trial 2 with value: 0.88

                 Model  Accuracy  Precision   Recall  F1 Score  Specificity  \
0  Stacking Classifier  0.883448   0.858967  0.83067  0.843211     0.941427   

   AUC     Kappa       MCC  
0    0  0.686803  0.689056  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>